In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import os
import snowflake.snowpark as snowpark
from snowflake.snowpark.session import Session
from snowflake.snowpark.functions import col, dateadd, current_date, date_trunc, lit

In [4]:
connection_parameters = {
    "account": "yva20138.us-west-2.privatelink",
    "user": "ben.pharris@dish.com",
    "authenticator": "externalbrowser",
    "insecure_mode":True,
}

session = Session.builder.configs(connection_parameters).create()

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://dish.okta.com/app/snowflake/exk9ewak82R5btDoW2p7/sso/saml?SAMLRequest=pZJNb%2BIwEIb%2FSuQ9x3YCCGoRKtqoWrRsoXxV7c0kBqw4dupxCP33a0JZdQ%2FtZW%2BR84zn8bwzvD2VKjgKC9LoBEWYokDozORS7xO0Xj2EAxSA4zrnymiRoHcB6HY0BF6qio1rd9AL8VYLcIG%2FSANrfySotpoZDhKY5qUA5jK2HP%2BeshhTVlnjTGYU%2BlTyfQUHENZ5w2tJDtLrHZyrGCFN0%2BCmg43dk5hSSugN8dQZ%2BXHlT%2F5NX%2FARod0z7wmPzz%2Fc7qS%2BjOA7re0FAvZztZqH89lyhYLxVfXeaKhLYZfCHmUm1ovpRQC8wctmHNOoM8A1hI2fXRjjysojd0JJXWDQptkpXojMlFXtfAvsv8hO5ESZvfRTmKQJqgqZbzZymz3%2Fmr3tis5kkB7TavbUjzf8IKfm9VRbqLpP2%2FXePZp9hoLNNeb4HPMEoBYTfQ7X%2BSMa98IoDiO6iiNGKaNd3O%2FRVxSkXlBq7trK6wtyCQdsCsdbM15V5K80EafiRjS8GMSL3tal5jmu%2BgTAkHPQ6LI7rO1uR%2F81kSH5fNXHTj76mCbp3CiZvQcPxpbcfZ1ihKP2RObhrkWZKLlU4zy3AsCnqZRp7q3wHglythaIjC5d%2F13%2B0R8%3D&Rela

In [3]:
# Sales Channel GAs

query = """
SELECT
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD,
    SUM(GROSS_ACTIVATION_QUANTITY) AS TOTAL_GROSS_ACTIVATION
FROM
    SBX.SBX_WRLSPPDSLSOPS.RADAR_GAS
WHERE
    ACTIVATION_DATE BETWEEN DATE_TRUNC('QUARTER', CURRENT_DATE) AND CURRENT_DATE
GROUP BY
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD
ORDER BY
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD
"""
ga_snowflake = session.sql(query).to_pandas()


NameError: name 'session' is not defined

In [ ]:

slsops_to_sales_channel = {
    'Direct Distribution Partner': 'Indirect',
    'Axiom Indirect': 'Digital',
    'Dish Internal': 'Digital',
    'Web Sales': 'Digital',
    'Amazon': 'Amazon',
    'Best Buy': 'National Retail',
    'National Retailer': 'National Retail',
    'WalMart': 'National Retail',
    'Target': 'National Retail',
    'The Kroger Co.': 'National Retail',
    'Cross Sell': 'Cross Sell',
    'Direct': 'Telesales',
    'Apple': 'Apple'
}

ga = ga_snowflake.dropna() 

ga['Channel'] = ga_snowflake['SLSOPS_CHANNEL_RECORD'].map(slsops_to_sales_channel)

ga.drop(['SLSOPS_CHANNEL_RECORD'], axis=1)


In [ ]:
spends = session.table('SBX.SBX_WRLSPPDSLSOPS.AH0027_AN_TEMP_MARKETING_PACING_DATAMART').to_pandas()

In [ ]:

spends['DATE'] = pd.to_datetime(spends['DATE'])

daily_sales_channel = spends.groupby(['CHANNEL', 'DATE'], as_index=False).sum().sort_values(['DATE','CHANNEL'])

assert daily_sales_channel[daily_sales_channel['CHANNEL'] != 'All']['SPEND'].sum() == daily_sales_channel[daily_sales_channel['CHANNEL'] == 'All']['SPEND'].sum()

#daily_sales_channel = daily_sales_channel[daily_sales_channel['CHANNEL'] != 'All']


In [ ]:
daily_sales_channel.to_csv("/Users/Ben.Pharris/Documents/project-dev/Ad Hoc/Atlas Reporting/spends.csv")

In [ ]:
query = 'SELECT * FROM SBX.SBX_WRLSPPDSLSOPS.RADAR_GAS limit 100'

conv_ex = session.sql(query).to_pandas()

conv_ex.to_csv("/Users/Ben.Pharris/Documents/project-dev/Ad Hoc/Atlas Reporting/conv_ex.csv")

In [ ]:
t = session.table("DISH_RETAIL_DL.ORDER_ORCHESTRATION.CUSTOMERORDER_PARSE")

In [ ]:
cutoff = dateadd("quarter", lit(-5), date_trunc("quarter", current_date()))

df_recent = t.filter(col("OO_LASTMODIFIED_DT_MT") >= cutoff)

df_recent.count()

In [5]:
query = "select * from SBX.SBX_WRLSPPDSLSOPS.AL0002_STORE_STATS where CLNDR_DT > '2025-11-22'"

doors = session.sql(query=query).to_pandas()

In [6]:
doors['CLNDR_DT'] = pd.to_datetime(doors['CLNDR_DT'])

In [ ]:
print(doors['CLNDR_DT'].max())

In [7]:
doors_plt = doors.groupby(['CLNDR_DT', 'REGION']).sum().reset_index()

In [8]:
doors_plt_subset = doors_plt[['CLNDR_DT', 'REGION', 'RAW_GROSS_ACTIVATIONS', 'DOOR_SWINGS']]

In [9]:
doors_plt_subset.to_parquet('doors_plt_subset.parquet', index=False)

In [ ]:
query = ""